# Solving a Tiny Poisson Linear System via Hand-Written QSVT

The goal of this notebook is deliberately modest: implement the core QSVT idea by hand for the smallest useful Poisson example, without calling a QSVT library.

We will solve a finite-difference linear system

$$
A u = f
$$

by preparing a normalized right-hand-side vector, constructing a block-encoding-like signal unitary for a normalized matrix $B=A/\alpha$, and applying a short QSP/QSVT phase sequence whose top-left block implements a polynomial $p(B)$.

For a linear-system solver, the useful polynomial is a scaled inverse:

$$
p(x) \approx \frac{\gamma}{x}.
$$

In this two-dimensional example we can choose the scale and polynomial so that the match is exact on the two eigenvalues of $B$.


In [ ]:
import numpy as np

np.set_printoptions(precision=10, suppress=True)


## 1. From the 1D Poisson Equation to a Matrix

Consider

$$
-u''(x)=f(x), \quad x \in (0,1), \quad u(0)=u(1)=0.
$$

Using second-order finite differences on two interior grid points gives, up to the common $1/h^2$ scale,

$$
A = \begin{pmatrix}2 & -1 \\ -1 & 2\end{pmatrix}.
$$

The global scale does not matter for a normalized quantum output state, so this toy demo uses the unscaled matrix above.


In [ ]:
def normalize(v):
    v = np.asarray(v, dtype=complex).reshape(-1)
    nrm = np.linalg.norm(v)
    if nrm == 0:
        raise ValueError('Cannot normalize the zero vector.')
    out = v / nrm
    return np.real_if_close(out, tol=1000)


def fidelity(psi, phi):
    psi = normalize(psi)
    phi = normalize(phi)
    return float(abs(np.vdot(psi, phi)) ** 2)


A = np.array([[2.0, -1.0], [-1.0, 2.0]])
f = np.array([1.0, 0.3])
f_state = normalize(f)
classical_solution = normalize(np.linalg.solve(A, f))

print('A =\n', A)
print('eigenvalues(A) =', np.linalg.eigvalsh(A))
print('|f> =', f_state)
print('normalized classical A^{-1}|f> =', classical_solution)


## 2. Normalize the Matrix Away from the Endpoint

QSP/QSVT has endpoint constraints. If we choose $\alpha=\|A\|_2=3$, then the largest eigenvalue of $B=A/\alpha$ is exactly $1$, which makes a low-degree inverse polynomial awkward.

For this toy implementation we choose

$$
\alpha = 4 > \|A\|_2.
$$

Then the eigenvalues of

$$
B = A/\alpha
$$

are $1/4$ and $3/4$, safely inside $(0,1)$.


In [ ]:
alpha = 4.0
B = A / alpha
B_eigs = np.linalg.eigvalsh(B)

print('alpha =', alpha)
print('B = A / alpha =\n', B)
print('eigenvalues(B) =', B_eigs)
print('||B||_2 =', np.linalg.norm(B, ord=2))


## 3. A Hand-Written Signal Unitary

For a Hermitian contraction $B$, one simple signal unitary is

$$
W(B) = \begin{pmatrix}
B & i\sqrt{I-B^2} \\
i\sqrt{I-B^2} & B
\end{pmatrix}.
$$

The top-left block is $B$, so this is a block-encoding of $B$. Because $B$ commutes with $\sqrt{I-B^2}$, this matrix is unitary.

This is the small-case version of the signal unitary that QSP/QSVT repeatedly queries.


In [ ]:
def matrix_sqrt_psd(M, tol=1e-12):
    vals, vecs = np.linalg.eigh(np.asarray(M, dtype=complex))
    if np.min(vals) < -tol:
        raise ValueError('Matrix is not positive semidefinite.')
    vals = np.clip(vals, 0.0, None)
    return (vecs * np.sqrt(vals)) @ vecs.conj().T


def hermitian_signal_unitary(B):
    B = np.asarray(B, dtype=complex)
    dim = B.shape[0]
    I = np.eye(dim, dtype=complex)
    defect = matrix_sqrt_psd(I - B @ B)
    return np.block([[B, 1j * defect], [1j * defect, B]])


W = hermitian_signal_unitary(B)
dim = B.shape[0]

block_error = np.linalg.norm(W[:dim, :dim] - B)
unitarity_error = np.linalg.norm(W.conj().T @ W - np.eye(2 * dim))

print('top-left block error =', block_error)
print('unitarity error =', unitarity_error)


## 4. The Degree-3 Inverse Polynomial

We want $p(x)$ to be proportional to $1/x$ on the two eigenvalues of $B$.

Choose

$$
\gamma = \frac{3}{32}.
$$

On $x=1/4$ and $x=3/4$, the cubic

$$
p(x) = \frac{5}{3}x - \frac{8}{3}x^3
$$

satisfies

$$
p(x)=\frac{\gamma}{x}.
$$

Therefore

$$
p(B)|f\rangle = \gamma B^{-1}|f\rangle = \gamma \alpha A^{-1}|f\rangle,
$$

which is the same normalized solution state as $A^{-1}|f\rangle$.


In [ ]:
gamma = 3.0 / 32.0


def inverse_polynomial(x):
    x = np.asarray(x)
    return (5.0 / 3.0) * x - (8.0 / 3.0) * x**3


for x in B_eigs:
    print(f'x = {x:.2f}, p(x) = {inverse_polynomial(x):.10f}, gamma / x = {gamma / x:.10f}')

xs = np.linspace(-1.0, 1.0, 2001)
print('sampled max |p(x)| on [-1, 1] =', np.max(np.abs(inverse_polynomial(xs))))


## 5. A Minimal QSP/QSVT Sequence

A scalar QSP step uses the signal matrix

$$
w(x)=\begin{pmatrix}
x & i\sqrt{1-x^2} \\
i\sqrt{1-x^2} & x
\end{pmatrix}
$$

and phase rotations

$$
S(\phi)=\begin{pmatrix}e^{i\phi}&0\\0&e^{-i\phi}\end{pmatrix}.
$$

The sequence

$$
S(\phi_0)w(x)S(\phi_1)w(x)S(\phi_2)w(x)S(\phi_3)
$$

has a top-left entry that is a degree-3 polynomial in $x$.

For this toy polynomial, the following phase angles implement $p(x)$ to numerical precision:

$$
\Phi = [0.00818588, -0.61547971, 0.61547971, 3.13340678].
$$

These are fixed low-degree phases for this demo, not a call to a phase-synthesis package.


In [ ]:
phases = np.array([0.00818588, -0.61547971, 0.61547971, 3.13340678])


def phase_operator(phi, dim):
    I = np.eye(dim, dtype=complex)
    return np.block([
        [np.exp(1j * phi) * I, np.zeros((dim, dim), dtype=complex)],
        [np.zeros((dim, dim), dtype=complex), np.exp(-1j * phi) * I],
    ])


def qsvt_sequence(signal_unitary, phases, dim):
    U = phase_operator(phases[0], dim)
    for phi in phases[1:]:
        U = U @ signal_unitary @ phase_operator(phi, dim)
    return U


U_qsvt = qsvt_sequence(W, phases, dim)
pB_from_qsvt = U_qsvt[:dim, :dim]

# Direct polynomial evaluation for comparison.
pB_direct = (5.0 / 3.0) * B - (8.0 / 3.0) * (B @ B @ B)

print('QSVT unitarity error =', np.linalg.norm(U_qsvt.conj().T @ U_qsvt - np.eye(2 * dim)))
print('||top-left block - p(B)|| =', np.linalg.norm(pB_from_qsvt - pB_direct))
print('top-left block p(B) from QSVT =\n', np.real_if_close(pB_from_qsvt, tol=1000))


## 6. Verify the Scalar Polynomial Implemented by the Phases

Before solving the linear system, check the scalar response of the phase sequence across $[-1,1]$.


In [ ]:
def scalar_signal(x):
    y = np.sqrt(max(0.0, 1.0 - x * x))
    return np.array([[x, 1j * y], [1j * y, x]], dtype=complex)


def scalar_qsp_response(x, phases):
    U = phase_operator(phases[0], 1)
    for phi in phases[1:]:
        U = U @ scalar_signal(x) @ phase_operator(phi, 1)
    return U[0, 0]


def drop_tiny_imaginary_parts(vector, tolerance=1e-8):
    vector = np.asarray(vector)
    if np.max(np.abs(vector.imag)) < tolerance:
        return vector.real
    return vector

sample_points = np.linspace(-1.0, 1.0, 401)
qsp_values = np.array([scalar_qsp_response(x, phases) for x in sample_points])
target_values = inverse_polynomial(sample_points)

print('max real error =', np.max(np.abs(qsp_values.real - target_values)))
print('max imaginary magnitude =', np.max(np.abs(qsp_values.imag)))

for x in [-1.0, -0.75, -0.25, 0.25, 0.75, 1.0]:
    print(f'x={x:5.2f}: QSP={scalar_qsp_response(x, phases): .10f}, target={inverse_polynomial(x): .10f}')


## 7. Apply QSVT and Postselect

The QSVT unitary acts on one signal ancilla plus the two-dimensional system. If the ancilla starts in $|0\rangle$, the top block after applying $U_{\Phi}$ is

$$
p(B)|f\rangle.
$$

Postselecting the signal ancilla on $|0\rangle$ gives the normalized solution state.


In [ ]:
initial_state = np.concatenate([f_state, np.zeros(dim, dtype=complex)])
final_state = U_qsvt @ initial_state
postselected_system = final_state[:dim]
success_probability = np.linalg.norm(postselected_system) ** 2
qsvt_solution = normalize(drop_tiny_imaginary_parts(postselected_system))

print('postselection probability =', success_probability)
print('QSVT solution state =', qsvt_solution)
print('classical solution state =', classical_solution)
print('fidelity =', fidelity(qsvt_solution, classical_solution))


## 8. What This Does and Does Not Implement

This notebook does implement the core QSVT mechanism for the easy Poisson case:

1. A hand-written signal unitary whose top-left block is $B=A/\alpha$.
2. A bounded odd polynomial that acts as a scaled inverse on the relevant eigenvalues.
3. A degree-3 QSP/QSVT phase sequence.
4. Postselection to obtain a normalized state proportional to $A^{-1}|f\rangle$.

It does not yet implement a general phase-angle synthesis algorithm. For larger matrices or arbitrary polynomial approximations, phase synthesis is the hard additional step. Here the phase sequence is fixed because the target polynomial and matrix spectrum are tiny.
